# [STARTER] Udaplay Project

## Part 01 - Offline RAG

In this part of the project, you'll build your VectorDB using Chroma.

The data is inside folder `project/starter/games`. Each file will become a document in the collection you'll create.
Example.:
```json
{
  "Name": "Gran Turismo",
  "Platform": "PlayStation 1",
  "Genre": "Racing",
  "Publisher": "Sony Computer Entertainment",
  "Description": "A realistic racing simulator featuring a wide array of cars and tracks, setting a new standard for the genre.",
  "YearOfRelease": 1997
}
```


### Setup

In [1]:
# Only needed for Udacity workspace

import importlib.util
import sys

# Check if 'pysqlite3' is available before importing
if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [2]:
import os
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(), override=True)
assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY is not set"
assert os.getenv("OPENAI_BASE_URL"), "OPENAI_BASE_URL is not set"
assert os.getenv("TAVILY_API_KEY"), "TAVILY_API_KEY is not set"

In [3]:
import os
import json
import chromadb
from chromadb.utils import embedding_functions
from dotenv import load_dotenv

In [4]:
# TODO
# Create a .env file in this folder (project/starter) with the following variables
# (copy .env.example from the repo root):
# OPENAI_API_KEY="voc-..."
# OPENAI_BASE_URL="https://openai.vocareum.com/v1"
# TAVILY_API_KEY="tvly-..."

### VectorDB Instance

In [5]:
# TODO: Instantiate your ChromaDB Client
# Keep this path: Part 2 loads the database from "chromadb" (inside project/starter)
chroma_client = chromadb.PersistentClient(path="chromadb")

### Collection

In [6]:
# TODO: Pick one embedding function
# Part 2 rebuilds this same function when it loads the collection,
# so if you pick something different, keep it identical in both notebooks.
embedding_fn = embedding_functions.OpenAIEmbeddingFunction(
     api_key_env_var="OPENAI_API_KEY",
     api_base=os.getenv("OPENAI_BASE_URL"),
 )

In [7]:
# TODO: Create a collection
# Keep this name: Part 2 loads the collection "udaplay"
collection = chroma_client.get_or_create_collection(
    name="udaplay",
    embedding_function=embedding_fn
)

### Add documents

In [8]:
# Run this notebook from project/starter: the game files are in ./games
data_dir = "games"

for file_name in sorted(os.listdir(data_dir)):
    if not file_name.endswith(".json"):
        continue

    file_path = os.path.join(data_dir, file_name)
    with open(file_path, "r", encoding="utf-8") as f:
        game = json.load(f)

    # You can change what text you want to index
    content = f"[{game['Platform']}] {game['Name']} ({game['YearOfRelease']}) - {game['Description']}"

    # Use file name (like 001) as ID
    doc_id = os.path.splitext(file_name)[0]

    collection.add(
        ids=[doc_id],
        documents=[content],
        metadatas=[game]
    )

In [10]:
import json

query = "action-packed games with fast-paced combat"

results = collection.query(
    query_texts=[query],
    n_results=3,
    include=["documents", "metadatas", "distances"],
)

games = []
for game_id, document, metadata, distance in zip(
    results["ids"][0],
    results["documents"][0],
    results["metadatas"][0],
    results["distances"][0],
):
    games.append({
        "game_id": game_id,
        "title": metadata["Name"],
        "description": metadata["Description"],
        "distance": float(distance),
        "metadata": metadata,
    })
print(collection.configuration)
print(json.dumps(games, indent=2))

{'hnsw': {'space': 'cosine', 'ef_construction': 100, 'ef_search': 100, 'max_neighbors': 16, 'resize_factor': 1.2, 'sync_threshold': 1000}, 'spann': None, 'embedding_function': <chromadb.utils.embedding_functions.openai_embedding_function.OpenAIEmbeddingFunction object at 0x1194f6c50>}
[
  {
    "game_id": "004",
    "title": "Marvel's Spider-Man",
    "description": "An open-world superhero game that lets players swing through New York City as Spider-Man, battling iconic villains.",
    "distance": 0.18963176012039185,
    "metadata": {
      "Name": "Marvel's Spider-Man",
      "YearOfRelease": 2018,
      "Genre": "Action-adventure",
      "Description": "An open-world superhero game that lets players swing through New York City as Spider-Man, battling iconic villains.",
      "Publisher": "Sony Interactive Entertainment",
      "Platform": "PlayStation 4"
    }
  },
  {
    "game_id": "013",
    "title": "Kinect Adventures!",
    "description": "A collection of mini-games designed t